# Classical Machine Learning Models using MFCC Features

## Introduction

After preparing the combined GTZAN and FMA Medium dataset and extracting MFCC feature vectors, the next step is to train classical Machine Learning baselines.

This notebook follows the existing project strategy:

- GTZAN and FMA Medium have already been harmonized into shared genre labels,
- the combined dataset has already been split into train, validation, and test subsets,
- MFCC features have already been extracted and saved as reusable CSV files.

The FMA Medium dataset is not treated as a separate external evaluation dataset here. The existing combined train / validation / test split is used consistently.

---

## Objectives

The main goals of this notebook are:

- load the prepared MFCC feature datasets,
- train reusable SVM and Random Forest baselines from `src/`,
- evaluate models on train and validation splits,
- select the best model using validation macro F1,
- evaluate the selected model once on the test split,
- and save model and evaluation artifacts.

---

## Evaluation Strategy

The validation split is used for model comparison and model selection.

The primary model selection metric is **macro F1 score** because the combined dataset is imbalanced. Macro F1 gives equal importance to every genre class, while accuracy and weighted F1 can be dominated by frequent classes.

After the best model is selected on validation data, it is evaluated once on the test split.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from src.models.random_forest_model import create_random_forest
from src.models.svm_model import create_svm
from src.training.evaluate import evaluate_model
from src.training.train import save_model, train_model
from src.utils.config import (
    MFCC_TRAIN_PATH,
    MFCC_VALIDATION_PATH,
    MFCC_TEST_PATH,
)

# Load MFCC Feature Datasets

The MFCC datasets were generated in the previous notebook and saved as CSV files.

Each row represents one audio recording. The `label` column contains the target genre, while columns starting with `mfcc_` contain the aggregated MFCC statistics.

In [ ]:
mfcc_train = pd.read_csv(MFCC_TRAIN_PATH)
mfcc_validation = pd.read_csv(MFCC_VALIDATION_PATH)
mfcc_test = pd.read_csv(MFCC_TEST_PATH)

print("Train:", mfcc_train.shape)
print("Validation:", mfcc_validation.shape)
print("Test:", mfcc_test.shape)

In [ ]:
mfcc_train.head()

# Prepare Features and Labels

All columns beginning with `mfcc_` are used as model inputs.

The label encoder is fitted on the training labels and reused for validation and test labels so that class indices remain consistent across all evaluations.

In [ ]:
feature_columns = [
    column
    for column in mfcc_train.columns
    if column.startswith("mfcc_")
]

X_train = mfcc_train[feature_columns].values
X_validation = mfcc_validation[feature_columns].values
X_test = mfcc_test[feature_columns].values

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(mfcc_train["label"])
y_validation = label_encoder.transform(mfcc_validation["label"])
y_test = label_encoder.transform(mfcc_test["label"])

class_names = list(label_encoder.classes_)

print("Feature count:", len(feature_columns))
print("Classes:", class_names)

In [ ]:
mfcc_train["label"].value_counts().loc[class_names]

# Create Baseline Models

Model definitions are provided by reusable factory functions in `src/models/`.

The SVM factory returns a pipeline containing `StandardScaler` and `SVC`, because SVM is sensitive to feature scale. Random Forest is created without scaling because tree-based models do not require standardized feature values.

In [ ]:
models = {
    "svm": create_svm(),
    "random_forest": create_random_forest(),
}

models

# Train and Evaluate Models

Training is handled through `train_model` from `src/training/train.py`.

Evaluation is handled through `evaluate_model` from `src/training/evaluate.py`, which computes accuracy, macro F1, weighted F1, a classification report, and a confusion matrix.

Both models are evaluated on the train and validation splits. Validation macro F1 is used for model selection.

In [ ]:
trained_models = {}
evaluation_results = []

for model_name, model in models.items():
    trained_model = train_model(
        model,
        X_train,
        y_train,
    )

    trained_models[model_name] = trained_model

    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("validation", X_validation, y_validation),
    ]:
        result = evaluate_model(
            trained_model,
            X_split,
            y_split,
            class_names,
            model_name=model_name,
            split=split_name,
        )

        evaluation_results.append(result)

# Train and Validation Summary

The summary table compares the models across train and validation splits.

Large gaps between train and validation metrics indicate overfitting. Validation macro F1 is the main criterion for choosing the final model.

In [ ]:
metrics_summary = pd.DataFrame([
    result["summary"]
    for result in evaluation_results
])

metrics_summary.sort_values(
    ["split", "macro_f1"],
    ascending=[True, False],
)

In [ ]:
validation_summary = (
    metrics_summary[metrics_summary["split"] == "validation"]
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

validation_summary

# Validation Classification Reports

The classification reports show precision, recall, and F1 score for each genre.

This is useful for identifying classes that are consistently confused or underperforming.

In [ ]:
validation_results = [
    result
    for result in evaluation_results
    if result["summary"]["split"] == "validation"
]

for result in validation_results:
    model_name = result["summary"]["model"]
    report = pd.DataFrame(result["classification_report"]).transpose()

    print("=" * 80)
    print(model_name)
    print("=" * 80)
    display(report)

# Select Best Model

The best model is selected by validation macro F1.

The test split is not used during model selection.

In [ ]:
best_model_name = validation_summary.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)
print("Validation macro F1:", validation_summary.iloc[0]["macro_f1"])

# Final Test Evaluation

After validation-based model selection, the selected model is evaluated once on the test split.

This produces the final classical Machine Learning baseline result for the current combined GTZAN + FMA dataset strategy.

In [ ]:
test_result = evaluate_model(
    best_model,
    X_test,
    y_test,
    class_names,
    model_name=best_model_name,
    split="test",
)

test_summary = pd.DataFrame([test_result["summary"]])
test_summary

In [ ]:
pd.DataFrame(test_result["classification_report"]).transpose()

# Test Confusion Matrix

The confusion matrix shows how predictions are distributed across the true genre classes.

Rows represent true labels and columns represent predicted labels.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=test_result["confusion_matrix"],
    display_labels=class_names,
)

display_matrix.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False,
)

ax.set_title(f"Test Confusion Matrix - {best_model_name}")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Save Model and Evaluation Artifacts

The selected model and evaluation outputs are saved for reuse in later stages.

The SVM artifact includes its fitted scaler because it is stored as a pipeline. If the selected model exposes a fitted scaler step, the scaler is also saved separately for inspection.

In [ ]:
artifact_dir = PROJECT_ROOT / "outputs" / "classical_ml"
model_dir = PROJECT_ROOT / "models" / "classical_ml"

artifact_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_classical_model.joblib"
label_encoder_path = model_dir / "label_encoder.joblib"
scaler_path = model_dir / "best_model_scaler.joblib"
metrics_summary_path = artifact_dir / "classical_ml_metrics_summary.csv"
test_confusion_matrix_path = artifact_dir / "test_confusion_matrix.png"
validation_reports_path = artifact_dir / "validation_classification_reports.csv"
test_report_path = artifact_dir / "test_classification_report.csv"

save_model(best_model, best_model_path)
save_model(label_encoder, label_encoder_path)

if hasattr(best_model, "named_steps") and "scaler" in best_model.named_steps:
    save_model(best_model.named_steps["scaler"], scaler_path)

final_metrics_summary = pd.concat(
    [metrics_summary, test_summary],
    ignore_index=True,
)

final_metrics_summary.to_csv(metrics_summary_path, index=False)

validation_report_tables = []

for result in validation_results:
    report = pd.DataFrame(result["classification_report"]).transpose()
    report.insert(0, "model", result["summary"]["model"])
    report.insert(1, "split", result["summary"]["split"])
    report.insert(2, "label", report.index)
    validation_report_tables.append(report.reset_index(drop=True))

pd.concat(validation_report_tables, ignore_index=True).to_csv(
    validation_reports_path,
    index=False,
)

test_report = pd.DataFrame(test_result["classification_report"]).transpose()
test_report.insert(0, "model", best_model_name)
test_report.insert(1, "split", "test")
test_report.insert(2, "label", test_report.index)
test_report.reset_index(drop=True).to_csv(test_report_path, index=False)

fig.savefig(
    test_confusion_matrix_path,
    dpi=150,
    bbox_inches="tight",
)

print("Saved best model:", best_model_path)
print("Saved label encoder:", label_encoder_path)

if hasattr(best_model, "named_steps") and "scaler" in best_model.named_steps:
    print("Saved scaler:", scaler_path)

print("Saved metrics summary:", metrics_summary_path)
print("Saved validation reports:", validation_reports_path)
print("Saved test report:", test_report_path)
print("Saved test confusion matrix:", test_confusion_matrix_path)

# Conclusion

This notebook trained and evaluated classical Machine Learning baselines using the reusable implementation in `src/`.

The following steps were completed:

- loaded train, validation, and test MFCC feature datasets,
- prepared feature matrices and encoded labels,
- created SVM and Random Forest models through `src/models/`,
- trained models through `src/training/train.py`,
- evaluated models through `src/training/evaluate.py`,
- selected the best model using validation macro F1,
- evaluated the selected model on the test split,
- and saved model and evaluation artifacts.

The resulting baseline provides a reproducible reference point for later Deep Learning experiments.